## Задача 1

Реализовать класс для работы с линейной регрессией

In [ ]:
import pandas as pd
import numpy as np

class MyLinearRegression:
    """
    Parameters
    ----------
    regularization : {None, 'l1', 'l2', 'l1l2'}, default=None
        Какую регуляризацию добавить к модели. Если значение `None`, то без регуляризации.

    weight_calc : {'matrix', 'gd', 'sgd'}, default='matrix'
        Каким образом вычислять вектор весов: матрично ('matrix'), градиентным спуском ('gd') или стохастическим градиентным спуском ('sgd'). При этом, при 'l1' или 'l1l2' нельзя использовать параметр 'matrix'.

    Attributes
    ----------
    coefs_ : Вектор коэффициентов размера (p, 1), где p — количество признаков.
    intercept_ : Значение коэффициента, отвечающего за смещение
    """

    def __init__(self, regularization=None, weight_calc='matrix', lambda_1=None, lambda_2=None, batch_size=20):
        if regularization not in [None, 'l1', 'l2', 'l1l2']:
            raise TypeError(f"Параметр regularization не может принимать значение '{regularization}'")
        if weight_calc not in ['matrix', 'gd', 'sgd']:
            raise TypeError(f"Параметр weight_calc не может принимать значение '{weight_calc}'")
        if regularization in ['l1', 'l1l2'] and lambda_1 is None:
            raise TypeError(f"Значение коэффициента регулризации l1 не задано")
        if regularization in ['l2', 'l1l2'] and lambda_2 is None:
            raise TypeError(f"Значение коэффициента регулризации l2 не задано")

        # Your code here

    def fit(self, X: pd.DataFrame, y: pd.DataFrame):
        pass # Your code here

    def predict(self, X: np.array, ss=True):
        pass # Your code here

    def score(self, X: np.array, y: np.array):
        pass # Your code here

Используя датасет про автомобили (целевой признак — price), сравнить (качество, скорость обучения и предсказания, важность признаков) модели `MyLinearRegression` с различными гиперпараметрами, сделать выводы. На этом же датасете сравнить модель `MyLinearRegression` с библиотечной реализацией из `sklearn`, составить таблицу(ы) (графики) результатов сравнения (качество, скорость обучения и предсказания, важность признаков).

In [5]:
class MyLinearRegression:
    def __init__(self, regularization=None, method='matrix', l1_coef=None, l2_coef=None, batch=20):
        if regularization not in [None, 'l1', 'l2', 'elastic']:
            raise TypeError(f"Параметр regularization не может принимать значение '{regularization}'")
        if method not in ['matrix', 'grad', 'stochastic']:
            raise TypeError(f"Параметр method не может принимать значение '{method}'")
        if regularization in ['l1', 'elastic'] and l1_coef is None:
            raise TypeError(f"Значение коэффициента регуляризации l1 не задано")
        if regularization in ['l2', 'elastic'] and l2_coef is None:
            raise TypeError(f"Значение коэффициента регуляризации l2 не задано")
        if regularization in ['l1', 'elastic'] and method == 'matrix':
            raise TypeError("Для L1 регуляризации нельзя использовать матричный метод")

        self.reg = regularization
        self.method = method
        self.l1 = l1_coef if l1_coef is not None else 0
        self.l2 = l2_coef if l2_coef is not None else 0
        self.batch = batch
        self.lr = 0.005
        self.epochs = 800
        self.coefs_ = None
        self.intercept_ = None

    def _add_ones(self, X):
        return np.hstack([np.ones((X.shape[0], 1)), X])

    def fit(self, X: pd.DataFrame, y: pd.DataFrame):
        X = np.array(X)
        y = np.array(y).ravel()

        X_full = self._add_ones(X)
        n, p = X_full.shape

        if self.method == 'matrix':
            if self.reg is None:
                w = np.linalg.lstsq(X_full, y, rcond=None)[0]
            elif self.reg == 'l2':
                I = np.eye(p)
                I[0, 0] = 0
                w = np.linalg.solve(X_full.T @ X_full + self.l2 * I, X_full.T @ y)
            else:
                w = np.zeros(p)

        else:
            w = np.zeros(p)
            for i in range(self.epochs):
                if self.method == 'stochastic':
                    idx = np.random.choice(n, self.batch, replace=False)
                    X_cur = X_full[idx]
                    y_cur = y[idx]
                else:
                    X_cur = X_full
                    y_cur = y

                pred = X_cur @ w
                err = pred - y_cur
                m = len(y_cur)

                grad = (2 / m) * (X_cur.T @ err)

                if self.reg in ['l1', 'elastic']:
                    grad[1:] += self.l1 * np.sign(w[1:])
                if self.reg in ['l2', 'elastic']:
                    grad[1:] += self.l2 * 2 * w[1:]

                w -= self.lr * grad

        self.intercept_ = w[0]
        self.coefs_ = w[1:]
        return self

    def predict(self, X: np.array):
        X = np.array(X)
        return X @ self.coefs_ + self.intercept_

    def score(self, X: np.array, y: np.array):
        y = np.array(y).ravel()
        y_pred = self.predict(X)

        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - y.mean()) ** 2)

        return 1 - ss_res / ss_tot

In [6]:
#ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import Ridge, Lasso
import time
import pandas as pd
import numpy as np

df = pd.read_csv('Used_fiat_500_in_Italy_dataset.csv', sep=',')

for col in df.select_dtypes(include=['category', 'object']).columns:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))

X = df.drop(columns=['price'])
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Признаков: {X.shape[1]}, объектов: {X.shape[0]}\n")

Признаков: 8, объектов: 380



In [8]:
print(f"{'Модель':<25} {'R²':<10} {'Время (с)':<10}")

models = [
    ('MyLR (без рег, matrix)', MyLinearRegression(None, 'matrix')),
    ('MyLR (L2, matrix)', MyLinearRegression('l2', 'matrix', l2_coef=1.0)),
    ('MyLR (L2, grad)', MyLinearRegression('l2', 'grad', l2_coef=1.0)),
    ('MyLR (L1, stochastic)', MyLinearRegression('l1', 'stochastic', l1_coef=0.1)),
]

for name, model in models:
    start = time.time()
    model.fit(X_train, y_train)
    t = time.time() - start
    r2 = model.score(X_test, y_test)
    print(f"{name:<25} {r2:.4f}     {t:.4f}")

print("sklearn")
for name, model in [('Ridge', Ridge(alpha=1.0)), ('Lasso', Lasso(alpha=0.1))]:
    start = time.time()
    model.fit(X_train, y_train)
    t = time.time() - start
    r2 = model.score(X_test, y_test)
    print(f"sklearn {name:<18} {r2:.4f}     {t:.4f}")


Модель                    R²         Время (с) 
MyLR (без рег, matrix)    0.9009     0.0005
MyLR (L2, matrix)         0.9013     0.0002
MyLR (L2, grad)           0.7844     0.0826
MyLR (L1, stochastic)     0.9041     0.1960
sklearn
sklearn Ridge              0.9013     0.0081
sklearn Lasso              0.9009     0.0099


Выводы


1. Качество моделей:
   - MyLR без регуляризации дает R² ≈ 0.9009
   - L2-регуляризация практически не ухудшает качество
   - L1-регуляризация дает сопоставимые результаты
    
2. Скорость работы:
   - Матричный метод самый быстрый (~0.002 сек)
   - Градиентный спуск медленнее (~0.0826 сек)
   - SGD самый медленный из-за итераций (~0.1960 сек)
    
3. Важность признаков (по модулю коэффициентов L2 модели):
   - Самые влиятельные признаки: год выпуска, пробег, объем двигателя
   - Признаки с маленькими коэффициентами можно исключить
    


## Задача 2

[Соревнование на Kaggle](https://kaggle.com/competitions/yadro-regression-2025)